# 第106章 物流延期风险预测项目

使用 Olist 巴西电商公开数据，以物流延期分类为主线，学习多表建模、预测时点、数据泄漏、类别不平衡和业务阈值。

## 项目背景

目标是在订单创建后预测是否会晚于预计日期签收。实际发货、实际签收和最终订单状态只能用于构造样本或标签，不能作为预测特征。

## 学习目标

- 理解订单、明细、客户和卖家表的粒度
- 构造订单级延期标签并排除事后字段
- 使用时间顺序划分模拟未来预测
- 比较概率基线、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片与特征重要性评价模型


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| order_id | 订单主键 | 最终保持一单一行 |
| customer/seller_state | 客户州/卖家州 | 下单时可用类别特征 |
| promise_days | 承诺时长 | 预计送达日减下单日 |
| goods/freight_value | 商品额/运费 | 订单级数值特征 |
| late | 是否延期 | 实际签收晚于预计日 |

## 数据质量检查清单

- 订单主键及明细一对多关系
- 聚合连接后订单唯一
- 日期缺失与承诺天数异常
- 已签收样本的选择口径
- 实际发货和签收字段泄漏
- 延期率随时间和地区变化


## 项目任务

1. 明确订单粒度、预测时点与延期标签
2. 审计四张原始数据表
3. 聚合明细并连接订单级样本
4. 清洗日期并构造延期标签
5. 探索类别不平衡和场景差异
6. 审计泄漏并按时间划分三组数据
7. 建立预处理Pipeline和概率基线
8. 比较逻辑回归与随机森林
9. 评价PR-AUC、阈值、Lift与错误切片
10. 解释特征重要性并总结局限


## 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 加载四表并审计数据粒度

先确认每张表的一行代表什么，再决定聚合和连接方式。


In [ ]:
import numpy as np
import pandas as pd

orders = pd.read_csv('/datasets/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_delivered_carrier_date'])
items = pd.read_csv('/datasets/olist_order_items_dataset.csv'); customers=pd.read_csv('/datasets/olist_customers_dataset.csv'); sellers=pd.read_csv('/datasets/olist_sellers_dataset.csv')
audit = pd.Series({'订单行':len(orders), '订单键重复':orders.order_id.duplicated().sum(), '明细行':len(items), '明细订单数':items.order_id.nunique(), '客户键重复':customers.customer_id.duplicated().sum(), '卖家键重复':sellers.seller_id.duplicated().sum()})
print(audit.to_string())


## 2. 聚合明细并连接订单级样本

先把一对多商品明细聚合到订单，再用validate检查连接基数。


In [ ]:
item_agg = items.groupby('order_id').agg(item_count=('order_item_id', 'size'), goods_value=('price', 'sum'), freight_value=('freight_value', 'sum'), primary_seller=('seller_id', 'first'), seller_count=('seller_id', 'nunique'))
order_level = (orders.merge(item_agg, on='order_id', validate='one_to_one').merge(customers[['customer_id', 'customer_state']], on='customer_id', validate='many_to_one').merge(sellers[['seller_id', 'seller_state']].rename(columns={'seller_id':'primary_seller'}), on='primary_seller', validate='many_to_one'))
assert order_level.order_id.is_unique
print('连接后订单:', len(order_level), '缺少实际签收:', order_level.order_delivered_customer_date.isna().sum()); display(order_level[['order_id', 'item_count', 'goods_value', 'freight_value', 'seller_count']].head())


## 3. 清洗样本并定义延期标签

标签来自签收结果；建模样本限定为有完整日期的已签收订单，并记录保留率。


In [ ]:
complete = (order_level.order_status.eq('delivered')&order_level.order_delivered_customer_date.notna()&order_level.order_estimated_delivery_date.notna())
model_df = order_level.loc[complete].copy(); model_df['promise_days']=(model_df.order_estimated_delivery_date-model_df.order_purchase_timestamp).dt.total_seconds()/86400
model_df = model_df.query('promise_days>0').copy(); model_df['late']=(model_df.order_delivered_customer_date>model_df.order_estimated_delivery_date).astype(int)
model_df['month']=model_df.order_purchase_timestamp.dt.month; model_df['weekday']=model_df.order_purchase_timestamp.dt.dayofweek; model_df=model_df.sort_values('order_purchase_timestamp').reset_index(drop=True)
print('建模订单:', len(model_df), '保留率:', f'{len(model_df)/len(orders):.1%}', '延期率:', f'{model_df.late.mean():.2%}')


## 4. 探索延期率与类别不平衡

观察月份、承诺时长和订单规模的差异，但不把描述性相关解释为延期原因。


In [ ]:
model_df['promise_group']=pd.qcut(model_df.promise_days,4, duplicates='drop')
month_rate = model_df.groupby('month').late.agg(['size', 'mean']); promise_rate=model_df.groupby('promise_group', observed=True).late.agg(['size', 'mean'])
print('月份延期率:\n', month_rate.round(3)); print('承诺时长分组延期率:\n', promise_rate.round(3)); print('多数类准确率:', f'{max(model_df.late.mean(),1-model_df.late.mean()):.2%}')


## 5. 泄漏审计与时间顺序划分

用早期订单训练、中期订单验证、晚期订单测试，模拟模型面对未来数据。


In [ ]:
num = ['item_count', 'goods_value', 'freight_value', 'seller_count', 'month', 'weekday', 'promise_days']; cat=['customer_state', 'seller_state']; features=num+cat
forbidden = ['order_delivered_customer_date', 'order_delivered_carrier_date', 'order_status']; assert not set(features)&set(forbidden)
train_end = int(len(model_df)*.64); val_end=int(len(model_df)*.80); train=model_df.iloc[:train_end]; val=model_df.iloc[train_end:val_end]; test=model_df.iloc[val_end:]
print('禁止字段:', forbidden); print('训练/验证/测试:', len(train), len(val), len(test)); print('延期率:',*[f'{part.late.mean():.2%}' for part in [train, val, test]])
print('训练截止:', train.order_purchase_timestamp.max(), '测试开始:', test.order_purchase_timestamp.min())


## 6. 预处理Pipeline与概率基线

类别编码、数值缩放和模型封装为统一流程；Dummy概率作为最低基线。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num', StandardScaler(), num)])
dummy = DummyClassifier(strategy='prior').fit(train[features], train.late); dummy_prob=dummy.predict_proba(val[features])[:,1]
print('验证集正类率:', round(val.late.mean(),3), 'Dummy PR-AUC:', round(average_precision_score(val.late, dummy_prob),3))


## 7. 比较逻辑回归与随机森林

在同一验证集上比较两个常见分类器，测试集仍保持未查看。


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {'逻辑回归':Pipeline([('prep', preprocess), ('model', LogisticRegression(max_iter=700, class_weight='balanced'))]), '随机森林':Pipeline([('prep', preprocess), ('model', RandomForestClassifier(n_estimators=160, min_samples_leaf=8, class_weight='balanced', n_jobs=-1, random_state=106))])}
rows = []
for name, model in models.items():
    model.fit(train[features], train.late); rows.append([name, average_precision_score(val.late, model.predict_proba(val[features])[:,1])])
validation = pd.DataFrame(rows, columns=['model', 'validation_PR_AUC']).sort_values('validation_PR_AUC', ascending=False); display(validation.round(3))
best_name = validation.iloc[0].model; dev=model_df.iloc[:val_end]; best_model=models[best_name].fit(dev[features], dev.late); probability=best_model.predict_proba(test[features])[:,1]


## 8. 测试集概率指标与阈值比较

PR-AUC是主指标，并比较不同Top-K比例下的精确率、召回率和Lift。


In [ ]:
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix

metrics = pd.Series({'ROC_AUC':roc_auc_score(test.late, probability), 'PR_AUC':average_precision_score(test.late, probability), 'LogLoss':log_loss(test.late, probability)})
ranked = pd.DataFrame({'actual':test.late.to_numpy(), 'probability':probability}).sort_values('probability', ascending=False); threshold_rows=[]
for share in [.05,.10,.20]:
    n = max(1, int(len(ranked)*share)); top=ranked.head(n); threshold_rows.append([f'{share:.0%}', top.probability.min(), top.actual.mean(), top.actual.sum()/ranked.actual.sum(), top.actual.mean()/ranked.actual.mean()])
threshold_table = pd.DataFrame(threshold_rows, columns=['Top比例', '概率阈值', 'Precision', 'Recall', 'Lift']); print(metrics.round(3).to_string()); display(threshold_table.round(3))
threshold=threshold_table.loc[threshold_table['Top比例']=='10%', '概率阈值'].iloc[0]; prediction=probability>=threshold; print('Top10%混淆矩阵:', confusion_matrix(test.late, prediction).tolist())


## 9. 错误类型与地区切片

区分漏判与误报，并检查模型在主要客户州的错误率是否一致。


In [ ]:
error_df = test[['customer_state', 'promise_days', 'item_count', 'late']].copy(); error_df['probability']=probability; error_df['prediction']=prediction
error_df['error_type']=np.select([(error_df.late==1)&(~error_df.prediction), (error_df.late==0)&error_df.prediction], ['漏判延期', '误报延期'], default='判断正确')
state_report=error_df.groupby('customer_state').agg(orders=('late', 'size'), late_rate=('late', 'mean'), mean_score=('probability', 'mean'), error_rate=('error_type', lambda x:(x!='判断正确').mean())).query('orders>=200').sort_values('error_rate', ascending=False)
print(error_df.error_type.value_counts()); display(state_report.head(12).round(3))


## 10. 特征解释与模型局限

用测试子样本计算置换重要性，并讨论数据缺失和预测边界。


In [ ]:
from sklearn.inspection import permutation_importance

sample_n = min(4000, len(test)); sample_idx=np.linspace(0, len(test)-1, sample_n, dtype=int)
permutation = permutation_importance(best_model, test.iloc[sample_idx][features], test.iloc[sample_idx].late, n_repeats=3, scoring='average_precision', random_state=106, n_jobs=-1)
importance = pd.Series(permutation.importances_mean, index=features).sort_values(ascending=False)
print('最佳模型:', best_name); print('置换重要性:\n', importance.round(4)); print('局限: 数据缺少距离、仓库节点、承运商和实时轨迹；地区特征的重要性不能解释为地区导致延期。')


## 结论与表达

- 多表建模必须先统一到订单粒度
- 标签可使用事后结果，但特征必须在预测时可获得
- 时间顺序划分比随机划分更接近未来预测
- 类别不平衡任务应联合观察PR-AUC、Recall和Lift


## 项目验收清单

- 完成四表粒度与连接审计
- 订单主键唯一且记录清洗口径
- 排除实际发货、签收和最终状态字段
- 比较Dummy和两个候选模型
- 完成阈值、错误切片和置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 Olist 巴西电商公开数据，以物流延期分类为主线，学习多表建模、预测时点、数据泄漏、类别不平衡和业务阈值。


### 你已经完成

- 理解订单、明细、客户和卖家表的粒度
- 构造订单级延期标签并排除事后字段
- 使用时间顺序划分模拟未来预测
- 比较概率基线、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片与特征重要性评价模型


### 建模流程速查

| 阶段 | 学习内容 |
| --- | --- |
| 步骤 1 | 明确订单粒度、预测时点与延期标签 |
| 步骤 2 | 审计四张原始数据表 |
| 步骤 3 | 聚合明细并连接订单级样本 |
| 步骤 4 | 清洗日期并构造延期标签 |
| 步骤 5 | 探索类别不平衡和场景差异 |
| 步骤 6 | 审计泄漏并按时间划分三组数据 |
| 步骤 7 | 建立预处理Pipeline和概率基线 |
| 步骤 8 | 比较逻辑回归与随机森林 |
| 步骤 9 | 评价PR-AUC、阈值、Lift与错误切片 |
| 步骤 10 | 解释特征重要性并总结局限 |


### 质量与结论提醒

- 订单主键及明细一对多关系
- 聚合连接后订单唯一
- 日期缺失与承诺天数异常
- 多表建模必须先统一到订单粒度
- 标签可使用事后结果，但特征必须在预测时可获得
- 时间顺序划分比随机划分更接近未来预测
- 类别不平衡任务应联合观察PR-AUC、Recall和Lift


### 学习检查

- [ ] 完成四表粒度与连接审计
- [ ] 订单主键唯一且记录清洗口径
- [ ] 排除实际发货、签收和最终状态字段
- [ ] 比较Dummy和两个候选模型
- [ ] 完成阈值、错误切片和置换重要性分析


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
